# Демо 2. Шаблони розмови і підготовка даних для SFT

В демо 1 донавчання моделі було повʼязане з заміною голови, а отже частковою зміною архітектури нейронів. В багатьох випадках при донавчанні LLM моделей архітектура не змінюється, а змінюється радше формат введених і виведених данних. Для цього потрібно зрозуміти, **у якому вигляді модель бачить діалог**.
В будь-якому випадку фізично на вхід моделі подається тензор з токенами, а за те як з цих токенів утворюються семантичні повідомлення відповідають **шаблони розмови** (chat templates).
Під час донавчання модель може вивчити новий шаблон розмови.

Донавчання LLM методом SFT виконується на діалогах, відформатованих шаблоном розмови.
У цьому ноутбуці розглянемо, що таке шаблон розмови і як підготувати дані у форматі діалогів;
саме донавчання виконаємо в демо 3.

План:
1. Завантажити base- та instruct-версії SmolLM3.
2. Подивитися, як шаблон розмови перетворює різні типи діалогів на текст.
3. Порівняти відповіді base- та instruct-моделі.
4. Перевірити режими міркування `/think` і `/no_think`.
5. Порівняти шаблони різних моделей.
6. Підготувати датасет для SFT у форматі `messages`.

Демо за мотивами вправ курсу [smol-course](https://huggingface.co/learn/smol-course) і розділу 11
[Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter11/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
# Встановлюємо потрібні бібліотеки; -q вимикає зайвий вивід під час встановлення
!pip install -q "transformers>=4.55.0" "datasets" "accelerate"

In [ ]:
import warnings

import torch
from datasets import get_dataset_split_names, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Памʼять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("Apple MPS")
else:
    device = "cpu"
    warnings.warn("Обчислення на CPU: генерація буде дуже повільною, потрібна GPU")

GPU: Tesla T4
Памʼять GPU: 15.6 GB


In [ ]:
# Детальна інформація про GPU NVIDIA: модель, зайнята й вільна памʼять
if device == "cuda":
    !nvidia-smi

Tue Nov 24 19:13:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Завантаження моделей

[**SmolLM3**](https://huggingface.co/blog/smollm3) — відкрита модель від Hugging Face на 3 млрд параметрів. Вона має дві версії:
- **Base** — лише після попереднього тренування: продовжує текст, але не веде діалог;
- **Instruct** (`SmolLM3-3B`) — донавчена виконувати інструкції й вести діалог.

In [ ]:
base_model_name = "HuggingFaceTB/SmolLM3-3B-Base"
instruct_model_name = "HuggingFaceTB/SmolLM3-3B"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)

# Ваги SmolLM3 збережено в bfloat16, але T4 не підтримує його апаратно,
# тому завантажуємо у float16.
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name, dtype=torch.float16, device_map="auto"
)
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_name, dtype=torch.float16, device_map="auto"
)
print("Моделі завантажено")

config.json:   0%|          | 0.00/943 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

... (12 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

Моделі завантажено


## 2. Як шаблон розмови форматує діалог

Діалог задається списком повідомлень із ролями:
- `system` — інструкції для моделі (роль, стиль, обмеження);
- `user` — повідомлення користувача;
- `assistant` — відповіді моделі.

Метод `apply_chat_template` перетворює цей список на рядок зі спеціальними токенами.

Параметр `add_generation_prompt`:
- `False` — для передачі в модель завершених діалогів, тобто для **тренувальних даних**;
- `True` — додає в кінець заголовок відповіді асистента, щоб модель почала відповідати; використовується для **інференсу**.

Для SmolLM3 у системне повідомлення можна додати прапорець `/think` або `/no_think`,
який вмикає чи вимикає режим міркування.

In [ ]:
# Різні типи діалогів для порівняння
conversations = {
    "simple_qa": [
        {"role": "system", "content": "/no_think"},
        {"role": "user", "content": "What is machine learning?"},
    ],
    "with_system": [
        {
            "role": "system",
            "content": "You are a helpful AI assistant specialized in explaining technical concepts clearly. /no_think",
        },
        {"role": "user", "content": "What is machine learning?"},
    ],
    "multi_turn": [
        {"role": "system", "content": "You are a math tutor. /no_think"},
        {"role": "user", "content": "What is calculus?"},
        {
            "role": "assistant",
            "content": "Calculus is a branch of mathematics that deals with rates of change and accumulation of quantities.",
        },
        {"role": "user", "content": "Can you give me a simple example?"},
    ],
    "reasoning_task": [
        {"role": "system", "content": "/think"},
        {
            "role": "user",
            "content": "Solve step by step: If a train travels 120 miles in 2 hours, what is its average speed?",
        },
    ],
}

for conv_type, messages in conversations.items():
    print(f"--- {conv_type.upper()} ---")

    # Завершений діалог: так виглядають тренувальні дані
    formatted_complete = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    # Діалог для інференсу: у кінці додано заголовок відповіді асистента
    formatted_prompt = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    print("Завершений діалог (add_generation_prompt=False):")
    print(formatted_complete)
    print("\nДля інференсу (add_generation_prompt=True):")
    print(formatted_prompt)
    print("\n" + "=" * 50 + "\n")

--- SIMPLE_QA ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 24 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>


Для інференсу (add_generation_prompt=True):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 24 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>
<|im_start|>assistant
<think>

</think>



--- WITH_SYSTEM ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 24 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant specialized in explaining technical concepts clearly.

<|im_s

**Що варто помітити:**
- Шаблон SmolLM3 сам додає до системного повідомлення блок метаданих (дата відсічення знань, поточна дата, режим міркування)
  і стандартну інструкцію, якщо власної немає. Модель бачить більше, ніж ми написали.
- Кожне повідомлення обгорнуте токенами `<|im_start|>роль` … `<|im_end|>` (формат ChatML).
- З `add_generation_prompt=True` рядок закінчується на `<|im_start|>assistant`: з цього місця модель починає генерувати.

## 3. Base проти Instruct

Ставимо обом моделям одне питання. Base-модель отримує сирий текст без шаблону,
instruct-модель — діалог, відформатований шаблоном розмови.

In [ ]:
def generate(model, tokenizer, text, max_new_tokens=150, temperature=0.7):
    """Генерує продовження тексту й повертає лише нові токени, без промпту."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Відкидаємо токени промпту: декодуємо лише те, що згенерувала модель
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [ ]:
test_prompt = "Explain me, what is quantum computing?"

# Base-модель: сирий текст без шаблону
print("🤖 BASE-МОДЕЛЬ:")
print("-" * 50)
print(generate(base_model, base_tokenizer, test_prompt))

# Instruct-модель: діалог, відформатований шаблоном розмови
instruct_messages = [
    {"role": "system", "content": "/no_think"},
    {"role": "user", "content": test_prompt},
]
instruct_formatted = instruct_tokenizer.apply_chat_template(
    instruct_messages, tokenize=False, add_generation_prompt=True
)
print("\n" + "=" * 50)
print("💬 INSTRUCT-МОДЕЛЬ:")
print("-" * 50)
print(generate(instruct_model, instruct_tokenizer, instruct_formatted))

🤖 BASE-МОДЕЛЬ:
--------------------------------------------------


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 How it’s different from normal computer? I am a beginner please give me a simple example.
Quantum computing, sometimes called quantum information processing, is the study of how to use quantum-mechanical phenomena, such as superposition and entanglement, to perform computation. A quantum computer is a device for performing such computation. Quantum computers are different from digital computers based on transistors. The concept of quantum computation was introduced by physicist Richard Feynman in 1982, and the basic requirements for a quantum computer were written down by Benioff in 1985. Quantum computers are not practical yet, but as of 2009, scientists and engineers around the world are racing to build one. A quantum computer would be able to

💬 INSTRUCT-МОДЕЛЬ:
--------------------------------------------------
Quantum computing is a type of computing that uses the principles of quantum mechanics to process information. Unlike traditional computers, which use bits to store and man

## 4. Режими міркування `/think` і `/no_think`

SmolLM3 — гібридна модель: у режимі `/think` вона спочатку генерує міркування в блоці `<think>…</think>`,
а потім відповідь; у режимі `/no_think` блок міркувань порожній.

In [ ]:
reasoning_prompts = [
    "What is 15 × 24? Show your work.",
    "A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?",
    "If I have $50 and spend $18.75 on lunch and $12.30 on a book, how much money do I have left?",
]

for thinking_flag in ["/no_think", "/think"]:
    print(f"===== Режим: {thinking_flag} =====\n")
    for i, prompt in enumerate(reasoning_prompts, 1):
        messages = [
            {"role": "system", "content": thinking_flag},
            {"role": "user", "content": prompt},
        ]
        formatted = instruct_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        # Нижча температура робить відповіді на арифметичні задачі стабільнішими
        answer = generate(instruct_model, instruct_tokenizer, formatted, max_new_tokens=300, temperature=0.3)
        print(f"Задача {i}: {prompt}")
        print(f"Відповідь: {answer}")
        print("\n" + "-" * 50 + "\n")

===== Режим: /no_think =====

Задача 1: What is 15 × 24? Show your work.
Відповідь: To solve 15 × 24, we can break it down into simpler steps.

First, let's consider 15 × 20. Since 15 × 20 is the same as 15 × (2 × 10), we can rewrite it as (15 × 2) × 10. 

15 × 2 is 30, and then multiplying by 10 gives us 300. So, 15 × 20 = 300.

Next, we need to find 15 × 4. This can be rewritten as (15 × 2) × 2. 

15 × 2 is 30, and then multiplying by 2 gives us 60. So, 15 × 4 = 60.

Now, we can add the two results together: 300 + 60 = 360.

Therefore, 15 × 24 = 360.

--------------------------------------------------

Задача 2: A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?
Відповідь: To determine how much flour is needed for 30 cookies, we can start by finding out how much flour is required for a single cookie. 

The recipe calls for 2 cups of flour for 12 cookies. To find the amount of flour per cookie, we divide the total amount of flour by the number 

## 5. У кожної моделі свій шаблон

Ролі `system`, `user`, `assistant` — це домовленість на рівні формату `messages`.
У токенах кожна модель записує їх по-своєму. Порівняємо шаблони кількох моделей.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi!"},
    {"role": "assistant", "content": "Hello! How can I help?"},
]

for name in ["HuggingFaceTB/SmolLM3-3B", "Qwen/Qwen3-0.6B", "microsoft/Phi-3.5-mini-instruct"]:
    tok = AutoTokenizer.from_pretrained(name)
    print(f"===== {name} =====")
    print(tok.apply_chat_template(messages, tokenize=False))
    print()

===== HuggingFaceTB/SmolLM3-3B =====
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 24 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful assistant.

<|im_start|>user
Hi!<|im_end|>
<|im_start|>assistant
Hello! How can I help?<|im_end|>




config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (7 outputs left out)

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

===== microsoft/Phi-3.5-mini-instruct =====
<|system|>
You are a helpful assistant.<|end|>
<|user|>
Hi!<|end|>
<|assistant|>
Hello! How can I help?<|end|>
<|endoftext|>



Прапорці режиму міркування також залежать від моделі. У Qwen3, наприклад,
режим вмикається параметром `enable_thinking` у `apply_chat_template`.

In [ ]:
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
question = [{"role": "user", "content": "What is 2 + 2?"}]

for enable_thinking in [True, False]:
    print(f"===== enable_thinking={enable_thinking} =====")
    print(qwen_tokenizer.apply_chat_template(
        question, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
    ))

===== enable_thinking=True =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant

===== enable_thinking=False =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant
<think>

</think>




### А що з base-моделлю?

Base-модель не донавчали на діалогах, тому шаблон у її токенізаторі або відсутній, або модель ніколи не бачила його токенів у ролі розмітки діалогу.
Перед SFT шаблон для base-моделі задають самостійно — найпростіше взяти його з instruct-версії тієї самої моделі.

In [ ]:
print("Шаблон є в base-токенізаторі:  ", base_tokenizer.chat_template is not None)
print("Шаблон є в instruct-токенізаторі:", instruct_tokenizer.chat_template is not None)

Шаблон є в base-токенізаторі:   False
Шаблон є в instruct-токенізаторі: True


## 6. Підготовка датасету для SFT

Для SFT дані зберігають у форматі `messages` — тобто у списку повідомлень із ролями.
Бібліотека TRL (`SFTTrainer`) сама застосовує до нього шаблон розмови під час тренування.

### SmolTalk2

**SmolTalk2** — датасет, на якому донавчали SmolLM3. Він складається з багатьох частин (splits) з різних джерел.
Список частин можна отримати без завантаження самих даних.

In [ ]:
splits = get_dataset_split_names("HuggingFaceTB/smoltalk2", "SFT")
print(f"Кількість частин: {len(splits)}")
for s in splits:
    print(" -", s)

README.md:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Кількість частин: 25
 - LongAlign_64k_Qwen3_32B_yarn_131k_think
 - OpenThoughts3_1.2M_think
 - aya_dataset_Qwen3_32B_think
 - multi_turn_reasoning_if_think
 - s1k_1.1_think
 - smolagents_toolcalling_traces_think
 - smoltalk_multilingual8_Qwen3_32B_think
 - smoltalk_systemchats_Qwen3_32B_think
 - table_gpt_Qwen3_32B_think
 - LongAlign_64k_context_lang_annotated_lang_6_no_think
 - Mixture_of_Thoughts_science_no_think
 - OpenHermes_2.5_no_think
 - OpenThoughts3_1.2M_no_think_no_think
 - hermes_function_calling_v1_no_think
 - smoltalk_multilingual_8languages_lang_5_no_think
 - smoltalk_smollm3_everyday_conversations_no_think
 - smoltalk_smollm3_explore_instruct_rewriting_no_think
 - smoltalk_smollm3_smol_magpie_ultra_no_think
 - smoltalk_smollm3_smol_rewrite_no_think
 - smoltalk_smollm3_smol_summarize_no_think
 - smoltalk_smollm3_systemchats_30k_no_think
 - table_gpt_no_think
 - tulu_3_sft_personas_instruction_following_no_think
 - xlam_traces_no_think
 - smoltalk_everyday_convs_reasoning_

In [ ]:
# Завантажуємо лише файл однієї частини через data_files: повний датасет займає десятки гігабайт,
# а load_dataset(..., split=...) спершу перебирає файли всіх частин і працює дуже довго
split = "smoltalk_everyday_convs_reasoning_Qwen3_32B_think"
everyday = load_dataset("HuggingFaceTB/smoltalk2", data_files=f"SFT/{split}-*.parquet", split="train")
print(everyday)
print()
for message in everyday[0]["messages"]:
    print(f"[{message['role']}] {message['content'][:200]}")

SFT/smoltalk_everyday_convs_reasoning_Qw(…): reconstructing file:   0%|          |  0.00B / 6.33MB            

SFT/smoltalk_everyday_convs_reasoning_Qw(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages', 'chat_template_kwargs', 'source'],
    num_rows: 2057
})

[user] Hi there
[assistant] <think>
Okay, the user sent "Hi there". That's a friendly greeting. I should respond in a welcoming way. Let me check the guidelines. I need to be helpful, keep the conversation going, and maybe ask h
[user] I'm looking for a healthy breakfast idea. What's a good option?
[assistant] <think>
Okay, the user is asking for a healthy breakfast idea. Let me think about what makes a breakfast healthy. It should be balanced, providing a mix of nutrients like protein, fiber, healthy fats,


Ця частина зібрана для режиму `/think`, тому відповіді асистента містять блок міркувань `<think>…</think>`.

Як видно з назви частини, відповіді в ній згенерувала модель Qwen3-32B. Такі **синтетичні дані** від сильнішої моделі —
поширений спосіб отримати тисячі прикладів для SFT без ручного написання відповідей. Помилки моделі-генератора при цьому переходять у дані,
тому синтетичні дані фільтрують.

### Перетворення власного датасету у формат `messages`

Більшість датасетів має інший формат, наприклад колонки «питання» і «відповідь».
Перетворимо датасет задач з математики **GSM8K** у формат `messages`.

In [ ]:
gsm8k = load_dataset("openai/gsm8k", "main", split="train[:100]")  # невелика вибірка для демо
print("Оригінальний приклад:", gsm8k[0])


def to_messages(examples):
    """Перетворює пари «питання — відповідь» на діалоги з ролями."""
    processed = []
    for question, answer in zip(examples["question"], examples["answer"]):
        processed.append([
            {"role": "system", "content": "You are a math tutor. Solve problems step by step."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ])
    return {"messages": processed}


gsm8k_messages = gsm8k.map(to_messages, batched=True, remove_columns=gsm8k.column_names)
print("\nУ форматі messages:", gsm8k_messages[0])

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Оригінальний приклад: {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


Map:   0%|          | 0/100 [00:00<?, ? examples/s]


У форматі messages: {'messages': [{'content': 'You are a math tutor. Solve problems step by step.', 'role': 'system'}, {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'role': 'user'}, {'content': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72', 'role': 'assistant'}]}


### Що побачить модель під час тренування

Застосуємо шаблон розмови до підготовленого прикладу. Саме цей рядок модель отримає під час SFT.

In [ ]:
text = instruct_tokenizer.apply_chat_template(
    gsm8k_messages[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,  # діалог завершений: відповідь асистента вже є
)
print(text)
print(f"Довжина в токенах: {len(instruct_tokenizer(text)['input_ids'])}")

<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 24 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a math tutor. Solve problems step by step.

<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72<|im_end|>

Довжина в токенах: 145


Шаблон знову додав метадані й режим `/think`, хоча в даних їх немає,
а відповідь асистента не містить блоку міркувань. Якщо донавчати на таких прикладах, модель вчитиметься
відповідати без міркувань навіть у режимі `/think`. Тому формат даних і шаблон мають узгоджуватися з тим,
як модель використовуватиметься після донавчання.

---
# Розділ 0.1. Експерименти для відповідей на питання

Відповіді наприкінці ноутбука


### До питання 16.

In [ ]:
prompt = "A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?"
for flag in ["/no_think", "/think"]:
    for limit in [300, 1500]:
        formatted = instruct_tokenizer.apply_chat_template(
            [{"role": "system", "content": flag}, {"role": "user", "content": prompt}],
            tokenize=False, add_generation_prompt=True)
        inputs = instruct_tokenizer(formatted, return_tensors="pt").to(instruct_model.device)
        torch.manual_seed(0)
        with torch.no_grad():
            out = instruct_model.generate(**inputs, max_new_tokens=limit, do_sample=True, temperature=0.3,
                                          pad_token_id=instruct_tokenizer.eos_token_id)
        new = out[0][inputs["input_ids"].shape[1]:]
        text = instruct_tokenizer.decode(new, skip_special_tokens=True)
        print(f"{flag:10} max_new_tokens={limit:<5} згенеровано {len(new):>4} токенів, "
              f"блок міркувань закрито: {'</think>' in instruct_tokenizer.decode(new)}, кінець: …{text[-120:]!r}")

/no_think  max_new_tokens=300   згенеровано  137 токенів, блок міркувань закрито: False, кінець: …'30):\n\n1/6 cup of flour per cookie * 30 cookies = 5 cups of flour.\n\nTherefore, 5 cups of flour are needed for 30 cookies.'
/no_think  max_new_tokens=1500  згенеровано  137 токенів, блок міркувань закрито: False, кінець: …'30):\n\n1/6 cup of flour per cookie * 30 cookies = 5 cups of flour.\n\nTherefore, 5 cups of flour are needed for 30 cookies.'
/think     max_new_tokens=300   згенеровано  300 токенів, блок міркувань закрито: False, кінець: …' 2 * 30. So 12x = 60. Then x = 60 / 12 = 5. So x is 5 cups. That seems right.\n\nWait, let me verify with the other method'
/think     max_new_tokens=1500  згенеровано  800 токенів, блок міркувань закрито: True, кінець: …'oth sides by 12:\n\n$$\nx = \\frac{60}{12} = 5\n$$\n\nThus, **5 cups of flour** are needed for 30 cookies.\n\n**Answer:** 5 cups.'


### До питання 20.

In [ ]:
from collections import Counter
print(Counter(s.split("_")[0] for s in splits).most_common())
print("think:", sum("think" in s and "no_think" not in s for s in splits), "| no_think:", sum("no_think" in s for s in splits))

[('smoltalk', 10), ('LongAlign', 2), ('OpenThoughts3', 2), ('table', 2), ('aya', 1), ('multi', 1), ('s1k', 1), ('smolagents', 1), ('Mixture', 1), ('OpenHermes', 1), ('hermes', 1), ('tulu', 1), ('xlam', 1)]
think: 10 | no_think: 15


# 0.1. Відповіді на питання 11–21

### 11. Як довго виконувалися клітинки ноутбуку 02_chat_templates? Чи були проблеми з запуском, які?

Ноутбук виконувався в Colab на Tesla T4. Найдовша клітинка — завантаження двох моделей SmolLM3-3B  у fp16: ≈ 7 хв (417 с), бо треба завантажити ≈ 12 GB ваг. Інші: встановлення бібліотек — 10 с, імпорти — 18 с, генерація base vs instruct — 15 с, 6 генерацій у режимах `/no_think` і `/think` — 59 с, порівняння шаблонів трьох моделей — 6 с, завантаження частин SmolTalk2 і GSM8K — по 1–3 с, мій додатковий експеримент з `/think` — 62 с. Разом ≈ 11 хв.

Проблеми:
* Colab відмовився запускати новий сеанс з повідомленням «Забагато активних сеансів» — довелося завершити сеанс попереднього ноутбука;
* Перший раз безкоштовний GPU був недоступний, тому довелося чекати;
* Попередження `Ignoring clean_up_tokenization_spaces=True for BPE tokenizer` та про `rope_parameters` у Phi-3.5 — не помилки, на результат не впливають. Обидві моделі 3B у fp16 (≈ 6 GB кожна) разом вмістилися в 15.6 GB памʼяті T4.

### 12. Для чого змінювати параметр add_generation_prompt в методі apply_chat_template() при використанні моделі? Як по згенерованим розмовам можна дізнатися значення цього параметру, яке було задано при виклику метода?

`add_generation_prompt` визначає, чи додавати в кінець відформатованого діалогу заголовок нової відповіді асистента. `False` — діалог «закритий»: так оформлюють тренувальні дані, де відповідь асистента вже є. `True` — у кінець дописується `<|im_start|>assistant`, тобто початок репліки асистента, з якого модель має почати генерацію під час інференсу. Якщо на інференсі поставити `False`, модель не знатиме, що зараз її черга, і може продовжити текст користувача або згенерувати новий `<|im_start|>user`.

Визначити значення за виводом, треба дивитися на кінець рядка: з `False` діалог закінчується останнім повідомленням (`…What is machine learning?<|im_end|>`), а з `True` — додатковим рядком `<|im_start|>assistant`. У SmolLM3 в режимі `/no_think` шаблон з `True` ще й одразу дописує порожній блок міркувань `<think>\n\n</think>` — так модель «підштовхують» відповідати без міркувань; у режимі `/think` (діалог REASONING_TASK) рядок закінчується лише `<|im_start|>assistant`, і модель сама почне з `<think>`.

### 13. Чим відрізняються ролі system від assistant в шаблонах розмов?

Роль `system` — це інструкції від розробника застосунку: хто така модель, яким стилем відповідати, які є обмеження, чи вмикати міркування (`/think`, `/no_think`). Зазвичай це одне повідомлення на початку діалогу, і модель навчена ставитися до нього як до правил, що діють на всю розмову. Роль `assistant` — це репліки самої моделі: її попередні відповіді в багатокроковому діалозі і та відповідь, яку вона має згенерувати. Під час SFT саме на токенах `assistant` рахується loss — модель вчиться їх генерувати, тоді як `system` і `user` є лише контекстом. У шаблоні SmolLM3 обидві ролі обгортаються однаково, але до `system` шаблон ще додає блок метаданих, а з `add_generation_prompt=True` в кінець ставиться заголовок `<|im_start|>assistant`, з якого модель починає генерацію.

### 14. Прокоментуйте вивід даних в секції 2 «Як шаблон розмови форматує діалог». Які секції було додано шаблонізатором розмов і для чого?

Хоча в повідомленнях ми передали лише `/no_think` чи короткий системний промпт, шаблон SmolLM3 перетворив системне повідомлення на структурований блок із двох секцій:
- `## Metadata` — `Knowledge Cutoff Date: June 2025` (дата, до якої модель має знання), `Today Date: 24 November 2026` (поточна дата, яку шаблон підставляє автоматично) і `Reasoning Mode: /no_think` або `/think`. Модель вчили на даних із такими метаданими, тому вона знає, яка сьогодні дата і чи треба міркувати; сам прапорець із тексту повідомлення шаблон прибирає і переносить сюди.
- `## Custom Instructions` — власний системний промпт. Якщо його немає (як у SIMPLE_QA, де системне повідомлення містить лише `/no_think`), шаблон підставляє стандартний: «You are a helpful AI assistant named SmolLM, trained by Hugging Face.» У режимі `/think` стандартна інструкція довша, бо вона детально описує, як міркувати.

У багатокроковому діалозі до попередньої відповіді асистента шаблон дописав порожній блок `<think>\n\n</think>` — так у тренувальних даних позначено, що в режимі `/no_think` модель не міркувала. Шаблон робить це, щоб формат на інференсі точно збігався з форматом, на якому модель донавчали; модель «бачить» більше, ніж ми написали.

### 15. Прокоментуйте різницю між Base і Instruct, проглянувши вивід в секції 3.

Base-модель на запит «Explain me, what is quantum computing?» не відповіла, а продовжила текст як форумне повідомлення: «How it's different from normal computer? I am a beginner please give me a simple example.» — тобто дописала ще питання від імені користувача, а далі видала уривок, схожий на статтю з енциклопедії («…introduced by physicist Richard Feynman in 1982 … as of 2009, scientists … are racing to build one»), із застарілими фактами й обірваний на півслові. Вона не розуміє ролей «користувач/асистент» і просто передбачає найімовірніше продовження інтернет-тексту.

Instruct-модель дала структуровану відповідь асистента: пояснила кубіти, суперпозицію, заплутаність і для яких задач квантові компʼютери швидші. Різниця — результат SFT і вирівнювання: instruct-версію донавчали на діалогах у форматі шаблону, тому вона розуміє, що треба відповідати на питання, і пише у стилі помічника. Обидві відповіді обірвалися на 150 токенах (`max_new_tokens=150`).

### 16. Чому в режимі /think при генерації було передано параметр max_new_tokens=300?

У режимі `/think` модель спочатку генерує міркування в блоці `<think>…</think>`, і вони займають сотні токенів, тож `max_new_tokens` обмежує загальну довжину генерації, щоб вона не тривала занадто довго: 3 задачі × 2 режими на T4 і так займають ≈ 1 хв. Значення 300 вибране як компроміс — для `/no_think` цього з запасом вистачає (повні відповіді займають ≈ 140 токенів), а для `/think` видно сам процес міркування. Але мій експеримент показує, що 300 токенів для `/think` замало:

| режим | max_new_tokens | згенеровано | блок міркувань закрито |
|---|---|---|---|
| `/no_think` | 300 | 137 | — (відповідь повна) |
| `/no_think` | 1500 | 137 | — |
| `/think` | 300 | **300** (обрізано) | ні |
| `/think` | 1500 | **800** | так, відповідь «5 cups» |

Для задачі з борошном модель у режимі `/think` використала 800 токенів: ≈ 650 на міркування і ще ≈ 150 на фінальну відповідь. З лімітом 300 у розділі 4 всі три відповіді в режимі `/think` обірвалися всередині блоку міркувань — фінальної відповіді користувач не побачив би зовсім. Тобто ліміт 300 у демо обраний, щоб економити час і показати, що міркування дорогі, а для реального використання режиму `/think` потрібно ≥ 1000–2000 токенів.

### 17. Чи впоралася модель з математичними задачами? Який би режим генерації Ви обрали для себе, якщо б Вам потрібна була допомога?

У режимі **`/no_think`** модель впоралася з усіма трьома задачами правильно й одразу: 15 × 24 = **360** (розклала на 15 × 20 + 15 × 4), для 30 печив потрібно **5 cups** борошна (2/12 = 1/6 чашки на печиво), залишок грошей **\$18.95** (50 − 31.05). Кожна відповідь — 100–150 токенів із поясненням кроків.

У режимі **`/think`** модель міркувала правильно (ті самі проміжні результати: 300 + 60 = 360, 1/6 × 30 = 5, 50 − 18.75 = 31.25), але через ліміт 300 токенів не встигла закінчити міркування й дати фінальну відповідь — усі три виводи обірвалися всередині `<think>` (наприклад, «…300 + 60. That should give me 360. Wait,»). Модель ще й повторно перевіряла себе («Wait, let me check that again»), що витрачає токени.

Для допомоги з такими простими арифметичними задачами я б обрала `/no_think`: відповідь правильна, у 2–5 разів коротша й швидша, а кроки розвʼязку модель і так показує. Режим `/think` варто вмикати для складних багатокрокових задач (олімпіадна математика, логічні задачі, програмування), де самоперевірка справді підвищує точність, — і тоді давати достатній ліміт токенів (≥ 1000).

### 18. Які відмінності в шаблонах розмов Ви помітили в секції 5?

Однаковий діалог (system + user + assistant) три моделі записують зовсім по-різному:
- SmolLM3 — формат ChatML (`<|im_start|>роль … <|im_end|>`), але системне повідомлення перетворюється на блок `## Metadata` (дата відсічення, поточна дата, `Reasoning Mode: /think`) + `## Custom Instructions`; за замовчуванням режим `/think`.
- Qwen3-0.6B — теж ChatML з тими самими `<|im_start|>`/`<|im_end|>`, але системне повідомлення передається як є, без метаданих; до відповіді асистента додається порожній блок `<think>\n\n</think>`. Режим міркування керується не текстом у system, а параметром `enable_thinking`: з `True` генерація починається з `<|im_start|>assistant` (модель сама відкриє `<think>`), з `False` шаблон одразу вставляє порожній `<think></think>`.
- Phi-3.5-mini-instruct — інший набір спецтокенів: `<|system|>`, `<|user|>`, `<|assistant|>`, кінець репліки `<|end|>`, а наприкінці `<|endoftext|>`; міркувань і метаданих немає.

Тобто ролі `system/user/assistant` — спільна домовленість на рівні формату `messages`, а в токенах кожна модель має власну розмітку. Тому дані для SFT зберігають у форматі `messages`, а шаблон конкретної моделі застосовують лише під час тренування/інференсу; і не можна, наприклад, надіслати Phi-3.5 текст, відформатований шаблоном Qwen. Також з розділу видно, що в base-токенізаторі SmolLM3 шаблону немає (`False`), а в instruct — є.

### 19. Для чого нам потрібно знати про шаблони розмов, якщо ми хочемо донавчити модель за допомогою SFT?

Під час SFT модель вчиться не просто відповідати, а відповідати у певному форматі токенів — саме в тому, у який шаблон розмови перетворив діалог. Якщо під час тренування і під час використання шаблон відрізнятиметься (інші спецтокени, інша розмітка ролей, інший режим міркування), модель побачить незнайомий формат і відповідатиме гірше або не зупинятиметься. Знання шаблону потрібне, щоб: (1) правильно підготувати дані у форматі `messages` і перевірити, що саме побачить модель (як у розділі 6 цього ноутбука); (2) задати шаблон для base-моделі, у якої його немає (найпростіше — взяти з instruct-версії); (3) рахувати loss лише на відповідях асистента (`assistant_only_loss`), для чого треба знати, де в шаблоні починається й закінчується відповідь; (4) узгодити режим міркування: якщо шаблон за замовчуванням додає `/think`, а відповіді в даних без міркувань, модель вивчить неправильну поведінку.

### 20. З яких даних було побудовано датасет HuggingFaceTB/smoltalk2?

SmolTalk2 — датасет, на якому Hugging Face донавчала SmolLM3. Список частин (клітинка розділу 6) показує, що він складається з **25 частин**, зібраних з різних джерел, — 10 для режиму `/think` (відповіді з міркуваннями) і 15 для `/no_think`:
- власні синтетичні діалоги Hugging Face SmolTalk (10 частин: everyday conversations, systemchats, smol-magpie-ultra, smol-rewrite, smol-summarize, explore-instruct-rewriting, multilingual 8 мов) — повсякденні розмови, переписування й підсумовування текстів, системні промпти;
- відкриті датасети міркувань: OpenThoughts3-1.2M, Mixture-of-Thoughts (science), s1k-1.1 — математика, наука, код із ланцюжками міркувань;
- виклик інструментів і агентів**: smolagents toolcalling traces, hermes-function-calling-v1, xLAM traces;
- довгий контекст: LongAlign-64k;
- таблиці: table-gpt; **багатомовність**: Aya dataset; **інструкції**: Tulu-3 SFT personas instruction following, OpenHermes-2.5; multi-turn reasoning з виконанням інструкцій.

Багато частин мають у назві `Qwen3_32B` — відповіді в них **синтетичні**, згенеровані сильнішою моделлю Qwen3-32B (як і показана в ноутбуці частина `smoltalk_everyday_convs_reasoning_Qwen3_32B_think`, 2 057 діалогів з блоком `<think>`). Тобто датасет поєднує відкриті людські/зібрані дані й синтетичні дані від сильнішої моделі, розподілені за режимами міркування.

### 21. Які модифікації даних відбуваються в секції "Перетворення власного датасету у формат messages"?

У розділі датасет GSM8K, де кожен приклад має дві колонки — `question` (умова задачі) і `answer` (розвʼязок з відповіддю), перетворюється у формат `messages`. Функція `to_messages` для кожної пари створює діалог із трьох повідомлень: `system` з інструкцією «You are a math tutor. Solve problems step by step.», `user` з текстом задачі і `assistant` з розвʼязком. `map(..., batched=True)` обробляє датасет пакетами, а `remove_columns=gsm8k.column_names` видаляє старі колонки, тож лишається одна колонка `messages`. Саме такий формат приймає `SFTTrainer`, який під час тренування сам застосує до нього шаблон розмови. Наступна клітинка показує, що шаблон SmolLM3 при цьому додає метадані й прапорець `/think`, хоча відповіді в GSM8K без блоку міркувань.